# HY.N.RE.E — Hybrid Narrative Retrieval Engine Demo

Demonstrates real-time narrative retrieval, **neighbor-chunk evidence window expansion**, and empirical **per-category ablation benchmarking** across 5 system configurations.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Add src directory to sys.path
sys.path.insert(0, str(Path.cwd() / "src"))
from retriever import HybridRetriever
from evaluator import run_full_ablation, load_json, EVAL_DIR
print("[OK] Environment loaded successfully.")

## 1. Query Execution & Ranked Passages (Hybrid + Rerank)

Retrieving top-3 passage results for a complex narrative query.

In [ ]:
retriever = HybridRetriever(strategy="scene")
query = "What does Elizabeth Bennet think of Mr. Darcy at the Netherfield ball?"

results = retriever.retrieve(query, method="hybrid_reranked", top_k=3, expand_neighbors=False)

print(f"Query: {query}\n" + "="*70)
for r in results:
    print(f"Rank {r['rank']} | Score: {r['score']:.4f} | Chapter: {r['chapter']} ({r['novel']})")
    print(f"Passage Snippet: {r['text'][:180]}...")
    print("-"*70)

## 2. Neighbor-Chunk Evidence Window Expansion (`expand_neighbors=True`)

Fetches contiguous preceding (`before`) and following (`after`) scene chunks within the same chapter.

In [ ]:
evidence_results = retriever.retrieve(query, method="hybrid_reranked", top_k=1, expand_neighbors=True)
ev = evidence_results[0]

print("=== EVIDENCE WINDOW (Target + Adjacent Chunks) ===")
print("\n[BEFORE CHUNK]")
print(ev['before']['text'][:150] + "..." if ev['before'] else "None (Chapter Start)")

print("\n[TARGET CHUNK (Rank 1)]")
print(ev['target']['text'][:200] + "...")

print("\n[AFTER CHUNK]")
print(ev['after']['text'][:150] + "..." if ev['after'] else "None (Chapter End)")

## 3. Empirical Ablation Results & Category Breakdown

Loading real empirical evaluation findings across all 5 system configurations.

In [ ]:
ablation_data = load_json(EVAL_DIR / "ablation_results.json")
df_overall = pd.DataFrame(ablation_data["overall_summary"]).T
df_by_category = pd.DataFrame(ablation_data["by_category_summary"]).T

print("=== OVERALL ABLATION METRICS ===")
display(df_overall)

print("\n=== PER-CATEGORY BREAKDOWN ===")
display(df_by_category)

## 4. Visualization — Retrieval System Performance Comparison

In [ ]:
plt.figure(figsize=(10, 5))
ax = df_overall[["Recall@5", "MRR", "NDCG@10"]].plot(kind="bar", figsize=(10, 5), colormap="viridis")
plt.title("HY.N.RE.E Retrieval Systems Comparison (Overall Metrics)", fontsize=14, fontweight="bold")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()